# Probability in Deep Learning

把本模块所有概念放进深度学习的真实场景：损失=负对数似然、softmax 的分布视角、Dropout 的不确定性、温度与校准、生成模型初探。


## 0. 环境配置与导入


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import torch
import matplotlib

matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Arial Unicode MS", "Microsoft YaHei", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)


## 1. 全景图：概率在深度学习的位置


| 概率概念 | 深度学习对应 |
|----------|--------------|
| 独立同分布 i.i.d. | 训练/测试数据假设 |
| 高斯噪声假设 | MSE 损失 |
| 伯努利 / 类别分布 | BCE / 交叉熵损失 |
| 条件分布 $P(y|x)$ | 分类器输出 |
| 期望 | 风险 = 期望损失 |
| 采样 | mini-batch、Dropout、生成 |
| KL 散度 | 变分推断（VAE）、蒸馏 |
| 极大似然 | 训练目标 NLL |


## 2. Softmax 的分布视角与温度


softmax 把 logits 变成概率分布，**温度 $T$** 控制分布的锐度：

$$p_i = \frac{e^{z_i/T}}{\sum_j e^{z_j/T}}$$

- $T \to 0$：分布逼近 one-hot（贪婪）
- $T = 1$：标准 softmax
- $T \to \infty$：分布趋近均匀（最大熵）


In [ ]:
logits = np.array([3.0, 1.0, 0.2])
def softmax_t(z, T):
    z = z / T
    e = np.exp(z - z.max())
    return e / e.sum()

for T in [0.3, 1.0, 5.0]:
    p = softmax_t(logits, T)
    H = -(p*np.log(p)).sum()
    print(f"T={T}: p={np.round(p,3)}  熵={H:.3f}")


In [ ]:
Ts = np.linspace(0.2, 5, 50)
plt.figure(figsize=(8, 4.5))
for i, z in enumerate(logits):
    plt.plot(Ts, [softmax_t(logits, T)[i] for T in Ts],
             label=f'类别 {i}（logit={z}）')
plt.xlabel('温度 T'); plt.ylabel('P(类别)')
plt.title('温度控制 softmax 的锐度')
plt.legend(); plt.grid(alpha=0.3)


## 3. Dropout 与 MC-Dropout 不确定性


训练时 Dropout 随机丢弃神经元 = **采样子网络**。推理时关闭 Dropout 得到确定输出——但如果推理时**保留** Dropout 并采样多次，输出的方差就度量了模型的不确定性（MC-Dropout，贝叶斯近似的实用版）。


In [ ]:
import torch.nn as nn
torch.manual_seed(0)
xs = torch.linspace(-3, 3, 150).unsqueeze(1)
ys = torch.sin(xs) + 0.1*torch.randn_like(xs)

model = nn.Sequential(
    nn.Linear(1, 64), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(64, 1))
opt = torch.optim.Adam(model.parameters(), lr=0.01)
for step in range(800):
    opt.zero_grad()
    loss = ((model(xs) - ys)**2).mean()
    loss.backward(); opt.step()

model.train()                                   # 保留 Dropout
with torch.no_grad():
    preds = torch.stack([model(xs) for _ in range(200)])
mean = preds.mean(0).numpy().ravel()
std = preds.std(0).numpy().ravel()
x0 = xs.numpy().ravel(); y0 = ys.numpy().ravel()

plt.figure(figsize=(8, 5))
plt.scatter(x0, y0, s=4, alpha=0.4, label='训练数据')
plt.plot(x0, np.sin(x0), 'k-', lw=2, label='真实 sin(x)')
plt.plot(x0, mean, 'r-', label='MC-Dropout 均值预测')
plt.fill_between(x0, mean-2*std, mean+2*std, color='red', alpha=0.2, label='±2σ 不确定性')
plt.legend(); plt.grid(alpha=0.3)
plt.title('MC-Dropout：不确定性在数据稀疏处自动变大')


## 4. 生成模型初探：ELBO 分解


生成模型想学 $p(x)$，但 $p(x) = \int p(x|z)p(z)\,dz$ 不可积。VAE 的做法：引入编码器 $q(z|x)$，把对数似然分解成两项（ELBO）：

$$\log p(x) \ge \underbrace{\mathbb{E}_{q(z|x)}\big[\log p(x|z)\big]}_{\text{重构项}} - \underbrace{D_{KL}\big(q(z|x)\,\|\,p(z)\big)}_{\text{正则项}}$$

- 重构项：$z$ 要能还原 $x$
- KL 项：编码分布 $q(z|x)$ 要贴近先验 $p(z)$


In [ ]:
# KL 项验证：q = N(μ, σ²) vs 先验 p = N(0,1)
def kl_gaussian(mu, sigma):
    return 0.5*(mu**2 + sigma**2 - 1 - 2*np.log(sigma))

# 蒙特卡洛对照
rng = np.random.default_rng(3)
mu, sigma = 1.2, 0.8
zs = rng.normal(mu, sigma, 200000)
log_q = -0.5*np.log(2*np.pi) - np.log(sigma) - (zs-mu)**2/(2*sigma**2)
log_p = -0.5*np.log(2*np.pi) - zs**2/2
mc_kl = (log_q - log_p).mean()

print(f"解析 KL = {kl_gaussian(mu, sigma):.4f}")
print(f"蒙特卡洛 KL = {mc_kl:.4f}")
print("→ KL 项 = 编码分布与先验的距离，训练时被压向 0")


## 5. 概率校准：预测概率可信吗


神经网络输出的"概率"未必等于真实频率（通常过于自信）。**校准**度量"预测 0.8 的样本是否真有 80% 为正类"。

- 温度缩放：用 $T$ 除 logits 再 softmax，在验证集上选最优 $T$（训练不动）
- 校准对决策、不确定性估计、医学/金融场景至关重要


In [ ]:
# 温度缩放演示：模拟一个"过于自信"的模型，用 T>1 校正
rng = np.random.default_rng(4)
# 模拟 logits：真实正类率 0.6，但模型输出概率偏高
y_true = (rng.random(10000) < 0.6).astype(float)
logits = np.where(y_true == 1,
                  rng.normal(0.9, 1.0, 10000),        # 正类 logit
                  rng.normal(-0.2, 1.0, 10000))       # 负类 logit

def calibrate(z, T):
    p = 1/(1+np.exp(-z/T))
    return p, (p > 0.5).astype(float)

for T in [1.0, 1.5, 2.0]:
    p, pred = calibrate(logits, T)
    acc = (pred == y_true).mean()
    conf = p.mean()
    print(f"T={T}: 准确率={acc:.3f}  平均置信度={conf:.3f}")
print("→ T>1 把过度自信的置信度拉回与准确率接近")


## 6. 概率统计模块总结


从"随机变量"到"深度学习"的完整链路：

$$\text{分布} \to \text{期望/方差} \to \text{联合/条件/贝叶斯} \to \text{独立/协方差} \to \text{极大似然} \to \text{信息论} \to \text{采样}$$

机器学习 = 假设一个分布族（损失函数）→ 在数据上最大化似然（训练）→ 用概率工具诊断不确定性（正则化/校准/贝叶斯近似）。


## 课后练习


1. **温度与熵**：证明 $T\to\infty$ 时 softmax 趋近均匀分布（熵 → log K）。
2. **MC-Dropout**：把 Dropout 概率从 0.2 改成 0.5，比较不确定性带的变化。
3. **ELBO**：写出 $D_{KL}(N(\mu,\sigma^2)\|N(0,1))$ 的推导过程（对 μ、σ 分别求导验证极值在 μ=0, σ=1）。
4. **校准**：用 5 节的模拟数据画出可靠性图（分桶预测概率 vs 实际频率）。
5. **综合**：用一个公式/一句话解释为什么"训练损失 = NLL"是概率视角的统一表述。
